In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
from pyspark.ml import PipelineModel
from pyspark.sql.functions import to_json, struct
# 1. Initialize Spark
spark = SparkSession.builder \
    .appName("FlightDelayInference") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# 2. Clean Schema
clean_schema = StructType([
    StructField("crs_dep_time", IntegerType(), True),
    StructField("dep_delay", DoubleType(), True),
    StructField("crs_arr_time", IntegerType(), True),
    StructField("arr_delay", DoubleType(), True),
    StructField("crs_elapsed_time", DoubleType(), True),
    StructField("distance", DoubleType(), True),
    StructField("temperature_2m", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("snowfall", DoubleType(), True),
    StructField("weather_code", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True),
    StructField("wind_gusts_10m", DoubleType(), True),
    StructField("surface_pressure", DoubleType(), True),
    StructField("pressure_msl", DoubleType(), True),
    StructField("is_weekend", IntegerType(), True),
    StructField("op_unique_carrier_index", DoubleType(), True),
    StructField("origin_index", DoubleType(), True),
    StructField("dest_index", DoubleType(), True)
])

# 3. Read CLEAN stream from beginning
print("📡 Listening to CLEAN Kafka topic 'flight-stream-clean'...")
clean_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "flight-stream-clean") \
    .option("startingOffsets", "earliest") \
    .load()

parsed_clean_stream = clean_stream.select(
    from_json(col("value").cast("string"), clean_schema).alias("data")
).select("data.*")

# 4. Load Model & Predict
print("🧠 Loading trained XGBoost model...")
ml_model = PipelineModel.load("hdfs://namenode:9000/flight_project/gold/xgboost_flight_delay_model")
predictions = ml_model.transform(parsed_clean_stream)



# 5. Output Comparison Table
final_output = predictions.select(
    col("crs_dep_time"),
    col("dep_delay").alias("Actual_Dep_Delay"),
    col("prediction").alias("Predicted_Arr_Delay"),
    col("arr_delay").alias("Actual_Arr_Delay")
)

# 6. Convert columns to JSON for Kafka
kafka_predictions = final_output.select(to_json(struct("*")).alias("value"))

print("🚀 Consumer Ready! Pushing live predictions to 'flight-predictions' Kafka topic...")
query = kafka_predictions.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "flight-predictions") \
    .option("checkpointLocation", "hdfs://namenode:9000/flight_project/checkpoints/predictions") \
    .start()



📡 Listening to CLEAN Kafka topic 'flight-stream-clean'...
🧠 Loading trained XGBoost model...
🚀 Consumer Ready! Pushing live predictions to 'flight-predictions' Kafka topic...


In [2]:
from kafka import KafkaConsumer
import json

# Look at the beginning of the topic
test_consumer = KafkaConsumer(
    'flight-predictions',
    bootstrap_servers=['kafka:9092'],
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    auto_offset_reset='earliest',
    consumer_timeout_ms=5000  # 5 second timeout
)

print("🔍 Pulling predictions from Kafka...")
print("-" * 50)

count = 0
for msg in test_consumer:
    print(msg.value)
    count += 1
    if count >= 5:  # Stop after seeing 5 predictions
        break

if count == 0:
    print("Still no messages. The Spark Consumer might not be writing to 'flight-predictions'.")
print("-" * 50)

🔍 Pulling predictions from Kafka...
--------------------------------------------------
{'crs_dep_time': 1727, 'Actual_Dep_Delay': 10.0, 'Predicted_Arr_Delay': 12.018414497375488, 'Actual_Arr_Delay': 10.0}
{'crs_dep_time': 1707, 'Actual_Dep_Delay': -8.0, 'Predicted_Arr_Delay': -13.249764442443848, 'Actual_Arr_Delay': -1.0}
{'crs_dep_time': 1229, 'Actual_Dep_Delay': -8.0, 'Predicted_Arr_Delay': -10.366236686706543, 'Actual_Arr_Delay': -16.0}
{'crs_dep_time': 1830, 'Actual_Dep_Delay': -4.0, 'Predicted_Arr_Delay': -10.654924392700195, 'Actual_Arr_Delay': -12.0}
{'crs_dep_time': 1929, 'Actual_Dep_Delay': -10.0, 'Predicted_Arr_Delay': -16.526187896728516, 'Actual_Arr_Delay': -14.0}
--------------------------------------------------


In [12]:
# Run this to stop the stream gracefully
for q in spark.streams.active:
    q.stop()

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 36096)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =